## Inspecting networks & runtime connect/disconnect

Two everyday skills close out single-host networking: **seeing** the wiring, and **rewiring** it live.

**Inspecting** — the commands you'll run constantly:

```bash
docker network ls                    # all networks
docker network inspect app-net       # full JSON: subnet, gateway, attached containers
docker port web                      # a container's published port mappings
docker inspect -f '{{.NetworkSettings.Networks.app-net.IPAddress}}' web
```

`network inspect` is the source of truth for "what's on this network and at which IP." A container attached to several networks has **several IPs — one per network.**

**Connect / disconnect at runtime.** A container can be on multiple networks at once, and you can change that **without restarting** it:

```bash
docker network create web-net db-net
docker run -d --name api --network web-net myorg/api
docker network connect db-net api        # api now on BOTH networks
docker network disconnect web-net api     # now only on db-net
```

Two patterns this enables:

- **Tiered isolation.** A reverse proxy on `web-net`, a database on `db-net`, and the API attached to **both**. The proxy can't see the DB, and the DB isn't reachable from the public side — segmentation by attachment, not firewall rules.
- **Drop-in debugging.** Attach a running container to a temporary network alongside a `tcpdump`/`wireshark` sidecar, then disconnect when done.

One detail: **`--network` at `docker run` takes only one network.** To start on several, run with one and `docker network connect` the rest before the app starts caring. This connect/disconnect flexibility is what makes user-defined bridges a real segmentation tool, not just name resolution.